# 01. 프롬프트 엔지니어링 실습

---

## 학습 목표

LLM의 출력 품질을 높이는 **핵심 프롬프트 기법 6가지**를 실습합니다.  
같은 질문에 다른 기법을 적용해 결과 차이를 직접 확인합니다.

## 다루는 기법

| 기법 | 핵심 아이디어 | 언제 쓰나? |
|---|---|---|
| **Zero-shot** | 예시 없이 바로 질문 | 간단한 일반 질문 |
| **Few-shot** | 예시(샷)를 프롬프트에 포함 | 특정 형식·스타일 유도 |
| **Chain-of-Thought** | 단계별 추론 유도 | 복잡한 분석·판단 |
| **역할 프롬프트** | LLM에게 페르소나 부여 | 전문 도메인 답변 |
| **구조화된 출력** | 특정 형식(JSON 등) 강제 | 파싱이 필요한 경우 |
| **프롬프트 템플릿** | 변수로 재사용 가능한 프롬프트 | 반복 작업 자동화 |

## 흐름

```
기법 학습 → 코드 실습 → 같은 질문으로 비교 실험 → 핵심 정리
```

In [1]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
import json
import sys
sys.path.append("..")

from common_config import llm_connect
model ="gpt-5.4"
temperature = 0
max_tokens = 2086

# LLM 객체 생성
llm = llm_connect(model=model)
# llm = llm_connect(model=model, temperature=temperature, max_tokens=max_tokens)


print("준비 완료")
print("실습에 사용할 공통 질문: '재택근무 전면 도입, 찬성해야 할까요?'")

준비 완료
실습에 사용할 공통 질문: '재택근무 전면 도입, 찬성해야 할까요?'


---

## 1. Zero-shot Prompting

**예시 없이** 바로 질문하는 가장 기본적인 방식입니다.  
LLM이 사전 학습 지식만으로 답변합니다.

```
사용자: [질문]
LLM:   [답변]
```

In [2]:
# ── Zero-shot: 예시 없이 바로 질문 ────────────────────────────
zero_shot_prompt = ChatPromptTemplate.from_messages([
    ("human", "{question}")
])

# 체인 생성 및 실행
chain = zero_shot_prompt | llm | StrOutputParser()
result = chain.invoke({"question": "재택근무 전면 도입, 찬성해야 할까요?"})

print("=== Zero-shot 결과 ===")
print(result)

=== Zero-shot 결과 ===
재택근무 전면 도입은 “무조건 찬성” 또는 “무조건 반대”로 보기보다, **조직의 특성과 업무 성격에 따라 신중히 판단해야 하는 사안**입니다. 다만 전반적으로는 **찬성할 만한 이유가 충분한 제도**라고 볼 수 있습니다.

## 찬성 이유
1. **업무 효율 향상 가능성**  
   출퇴근 시간이 사라져 직원의 피로가 줄고, 집중할 수 있는 환경이 확보되면 생산성이 높아질 수 있습니다.

2. **직원 만족도 증가**  
   일과 생활의 균형이 좋아져 스트레스가 줄고, 직무 만족도와 장기 근속 가능성이 높아질 수 있습니다.

3. **비용 절감**  
   회사는 사무실 운영비를 줄일 수 있고, 직원은 교통비·식비 등 부수 비용을 아낄 수 있습니다.

4. **인재 확보에 유리**  
   지역에 상관없이 인재를 채용할 수 있어 채용 경쟁력이 높아집니다.

## 반대 또는 우려되는 점
1. **소통과 협업의 어려움**  
   대면 소통이 줄어들면 오해가 생기거나 협업 속도가 느려질 수 있습니다.

2. **조직 문화 약화**  
   구성원 간 유대감이 떨어지고, 신입 직원은 조직에 적응하기 더 어려울 수 있습니다.

3. **업무 관리 문제**  
   직무에 따라 성과 측정이 어렵거나, 자기관리가 부족한 경우 생산성이 오히려 떨어질 수 있습니다.

4. **업무와 사생활의 경계 모호**  
   집이 곧 일터가 되면서 오히려 더 오래 일하거나 휴식이 부족해질 가능성도 있습니다.

## 결론
재택근무 전면 도입은 **효율성, 만족도, 비용 측면에서 장점이 크기 때문에 기본적으로 찬성할 수 있지만**, 모든 조직에 일률적으로 적용하는 것은 위험할 수 있습니다.  
따라서 가장 현실적인 방향은 **전면 도입 자체를 목표로 하되, 직무별 차이를 고려하거나 하이브리드 방식으로 보완하는 것**입니다.

원하시면


---

## 2. Few-shot Prompting

프롬프트 안에 **예시(shots)** 를 포함해서 원하는 출력 형식을 보여줍니다.  
LLM이 패턴을 학습해 같은 형식으로 답변합니다.

```
시스템: 예시1: Q→A, 예시2: Q→A  ← 패턴 학습
사용자: [새 질문]
LLM:   [예시와 같은 형식의 답변]  ← 패턴 따라함
```

In [3]:
# ── Few-shot: 예시 2개를 포함해서 출력 형식을 유도 ────────────
few_shot_prompt = ChatPromptTemplate.from_messages([
    ("system", """아래 예시와 똑같은 형식으로 답변하세요.

예시 1)
질문: 주 4일 근무제를 도입해야 할까요?
답변:
✅ 찬성 근거
- 직원 삶의 질 향상으로 생산성 증가
- 채용 경쟁력 강화

❌ 반대 근거
- 업무량 미감소 시 오히려 번아웃 위험
- 서비스 연속성 유지 어려움

💡 결론: 업종·직무별로 차등 적용하는 파일럿 테스트를 권장합니다.

예시 2)
질문: 사내 카페테리아를 없애야 할까요?
답변:
✅ 찬성 근거
- 운영 비용 절감
- 외부 다양한 식사 옵션 활용 가능

❌ 반대 근거
- 직원 식사 복지 감소
- 점심 시간 이탈로 협업 기회 감소

💡 결론: 직원 수요 조사 후 결정하되, 식대 지원으로 보완을 고려하세요."""),
    ("human", "질문: {question}")
])

# 체인 생성 및 실행
chain = few_shot_prompt | llm | StrOutputParser()
result = chain.invoke({"question": "재택근무 전면 도입, 찬성해야 할까요?"})

print("=== Few-shot 결과 ===")
print(result)

=== Few-shot 결과 ===
답변:
✅ 찬성 근거
- 출퇴근 시간 절감으로 업무 만족도와 효율 향상
- 지역 제한 없이 우수 인재 채용 가능
- 사무실 운영비 절감

❌ 반대 근거
- 대면 협업과 조직문화 형성 약화 가능
- 업무 집중도와 성과 관리의 편차 발생
- 보안 및 정보 유출 리스크 증가

💡 결론: 전면 도입보다는 직무 특성에 따라 재택과 출근을 병행하는 하이브리드 방식이 현실적입니다.


---

## 3. Chain-of-Thought (CoT) Prompting

LLM에게 **단계별로 생각하도록** 유도합니다.  
복잡한 추론, 계획, 분석이 필요한 질문에 효과적입니다.

```
기본:  질문 → 바로 답변 (shallow)
CoT:  질문 → 단계1 → 단계2 → 단계3 → 결론 (deep)
```

In [4]:
# ── Chain-of-Thought: 단계별 사고 과정 명시 ───────────────────
cot_prompt = ChatPromptTemplate.from_messages([
    ("system", """질문에 답하기 전에 반드시 다음 4단계로 생각하고, 각 단계를 명시하세요.

[1단계: 문제 파악] 무엇을 묻고 있는가?
[2단계: 이해관계자 분석] 누가 영향을 받는가?
[3단계: 장단점 검토] 도입/미도입 시 각각 어떤 결과가?
[4단계: 최종 판단] 근거를 바탕으로 한 결론은?"""),
    ("human", "{question}")
])

# 체인 생성 및 실행
chain = cot_prompt | llm | StrOutputParser()
result = chain.invoke({"question": "재택근무 전면 도입, 찬성해야 할까요?"})

print("=== Chain-of-Thought 결과 ===")
print(result)

=== Chain-of-Thought 결과 ===
[1단계: 문제 파악]  
질문은 “재택근무를 전면 도입하는 것에 찬성할 것인가”에 대한 판단입니다. 즉, 일부가 아니라 조직 전체 차원에서 재택근무를 기본 근무 방식으로 바꾸는 것이 바람직한지 묻고 있습니다.

[2단계: 이해관계자 분석]  
영향을 받는 주체는 다음과 같습니다.  
- 직원: 출퇴근 부담, 워라밸, 업무 몰입도, 고립감 등에 영향  
- 기업/경영진: 생산성, 관리 방식, 비용 절감, 조직문화 유지에 영향  
- 팀 관리자: 성과 관리, 소통 방식, 협업 효율 변화  
- 고객/거래처: 응대 속도, 서비스 품질 변화 가능  
- 사회 전반: 교통 혼잡, 지역 상권, 탄소 배출 등에도 영향

[3단계: 장단점 검토]  
1. 전면 도입에 찬성할 이유  
- 출퇴근 시간이 줄어 직원 만족도와 삶의 질이 높아질 수 있습니다.  
- 사무실 운영비용을 절감할 수 있습니다.  
- 지역에 상관없이 인재 채용이 가능해집니다.  
- 집중이 잘 되는 직무에서는 생산성이 오를 수 있습니다.  
- 교통량 감소와 환경 측면에서도 긍정적입니다.

2. 전면 도입의 문제점  
- 협업, 즉흥적 소통, 조직문화 형성이 약해질 수 있습니다.  
- 신입사원 교육이나 멘토링이 어려워질 수 있습니다.  
- 직원에 따라 자기관리 능력과 집 환경 차이로 생산성 격차가 커질 수 있습니다.  
- 고립감, 소속감 저하, 장시간 근무 같은 부작용이 생길 수 있습니다.  
- 보안 문제나 데이터 관리 위험도 커질 수 있습니다.

3. 전면 도입하지 않을 경우  
- 대면 협업과 조직문화 유지에는 유리합니다.  
- 다만 출퇴근 부담과 사무실 비용 문제는 계속 남습니다.  
- 유연성을 중시하는 인재 확보 경쟁에서 불리할 수 있습니다.

[4단계: 최종 판단]  
저는 “전면 도입”


---

## 4. 역할(Role) 프롬프트

LLM에게 **특정 전문가 페르소나**를 부여합니다.  
같은 질문이라도 역할에 따라 완전히 다른 관점으로 답변합니다.

```
역할 없음:  중립적·일반적 답변
HR 전문가:  사람 중심의 조직 관점
CFO:       비용·효율 중심의 재무 관점
```

In [5]:
# ── 역할 프롬프트: 같은 질문, 다른 역할 ──────────────────────
def make_role_chain(role_description: str):
    """역할 설명을 받아 전문가 체인을 생성한다"""
    prompt = ChatPromptTemplate.from_messages([
        ("system", role_description),
        ("human", "{question}")
    ])
    chain = prompt | llm | StrOutputParser()
    return chain


# 역할 1: HR 디렉터
hr_chain = make_role_chain("""
당신은 15년 경력의 HR 디렉터입니다.
직원 복지, 조직 문화, 인재 유지에 최우선 가치를 둡니다.
답변은 구성원 관점에서 3~4문장으로 핵심만 짚어주세요.
""")

# 역할 2: 재무이사(CFO)
cfo_chain = make_role_chain("""
당신은 10년 경력의 재무이사(CFO)입니다.
모든 결정을 비용-효익 분석으로 접근합니다.
답변은 수치 근거를 중심으로 3~4문장으로 핵심만 짚어주세요.
""")

question = "재택근무 전면 도입, 찬성해야 할까요?"

print("=== HR 디렉터 관점 ===")
print(hr_chain.invoke({"question": question}))

print("\n=== CFO 관점 ===")
print(cfo_chain.invoke({"question": question}))

=== HR 디렉터 관점 ===
전면 재택근무는 구성원 만족도와 몰입도를 높이고, 출퇴근 부담을 줄여 인재 유지에 긍정적일 수 있습니다. 다만 협업 밀도, 온보딩, 소속감 약화 같은 조직문화 리스크를 함께 봐야 합니다. 그래서 무조건 찬반보다, 직무 특성에 따라 하이브리드·선택형 모델로 설계하는 것이 현실적입니다. 핵심은 “어디서 일하느냐”보다 성과 기준, 소통 방식, 리더십 운영체계를 먼저 갖추는 것입니다.

=== CFO 관점 ===
조건부 찬성입니다. 사무실 비용은 1인당 연 300만~800만원 절감 가능하지만, 생산성이 5%만 하락해도 연봉 6,000만원 기준 인당 300만원 손실이라 절감 효과가 상쇄될 수 있습니다. 반면 이직률이 5%p만 낮아져도 채용·온보딩 비용(연봉의 20~30%) 절감 효과가 커서 총비용 측면에선 플러스일 가능성이 높습니다. 따라서 “전면 즉시 도입”보다 3~6개월 파일럿으로 생산성, 이직률, 협업지표를 측정해 순효익이 확인될 때 확대하는 것이 CFO 관점에서 합리적입니다.


---

## 5. 구조화된 출력 (Structured Output)

LLM의 출력을 **JSON 등 특정 형식으로 강제**합니다.  
후처리(파싱, DB 저장, API 응답)가 필요할 때 사용합니다.

```
일반 출력: 자연어 문장 → 파싱 불가
구조화:   JSON/YAML → 자동 파싱 가능 → 시스템 연동 가능
```

In [6]:
# ── 구조화된 출력: JSON 형식 강제 ────────────────────────────
structured_prompt = ChatPromptTemplate.from_messages([
    ("system", """반드시 아래 JSON 형식으로만 답변하세요. 다른 텍스트는 절대 포함하지 마세요.

  - 출력형식
  "topic": "주제",
  "pros": ["찬성 근거1", "찬성 근거2", "찬성 근거3"],
  "cons": ["반대 근거1", "반대 근거2"],
  "recommendation": "최종 권고사항",
  "confidence": 확신도(1~10)
"""),
    ("human", "다음 주제를 분석해줘: {question}")
])

# 체인 생성 및 실행
chain = structured_prompt | llm | StrOutputParser()
raw_output = chain.invoke({"question": "재택근무 전면 도입"})

print("=== LLM 원본 출력 ===")
print(raw_output)

# JSON 파싱
print("\n=== 파싱 후 활용 ===")
try:
    data = json.loads(raw_output)
    print(f"주제     : {data['topic']}")
    print(f"확신도   : {data['confidence']}/10")
    print(f"권고사항 : {data['recommendation']}")
    print(f"찬성 근거: {', '.join(data['pros'])}")
except json.JSONDecodeError as e:
    print(f"파싱 실패: {e}")
    print("→ 팁: 모델이 JSON을 못 지키면 출력에서 JSON 부분만 추출하거나 재시도 로직을 추가하세요.")

=== LLM 원본 출력 ===
{
  "topic": "재택근무 전면 도입",
  "pros": [
    "출퇴근 시간이 사라져 직원의 업무 만족도와 삶의 질이 향상될 수 있다",
    "사무실 임대료와 운영비 등 고정비를 절감할 수 있다",
    "지역 제한 없이 더 넓은 인재풀에서 채용할 수 있다"
  ],
  "cons": [
    "대면 협업 감소로 의사소통 오류와 팀 결속력 약화가 발생할 수 있다",
    "직무에 따라 성과 관리, 보안 통제, 업무 몰입 유지가 더 어려워질 수 있다"
  ],
  "recommendation": "전면 도입은 직무 특성과 조직 문화에 따라 신중히 결정해야 하며, 가능하다면 보안 체계·성과관리 기준·협업 프로세스를 먼저 정비한 뒤 단계적으로 도입하는 것이 바람직하다",
  "confidence": 9
}

=== 파싱 후 활용 ===
주제     : 재택근무 전면 도입
확신도   : 9/10
권고사항 : 전면 도입은 직무 특성과 조직 문화에 따라 신중히 결정해야 하며, 가능하다면 보안 체계·성과관리 기준·협업 프로세스를 먼저 정비한 뒤 단계적으로 도입하는 것이 바람직하다
찬성 근거: 출퇴근 시간이 사라져 직원의 업무 만족도와 삶의 질이 향상될 수 있다, 사무실 임대료와 운영비 등 고정비를 절감할 수 있다, 지역 제한 없이 더 넓은 인재풀에서 채용할 수 있다


---

## 6. 프롬프트 템플릿 (재사용 가능한 팩토리)

**변수화된 프롬프트**를 만들어 다양한 도메인에 재사용합니다.  
같은 구조의 프롬프트를 반복해서 만들 때 효율적입니다.

```python
# 나쁜 예: 비슷한 프롬프트를 계속 복붙
legal_prompt   = "당신은 변호사입니다... {question}"
medical_prompt = "당신은 의사입니다... {question}"

# 좋은 예: 팩토리 함수로 생성
make_expert_chain(domain="법률", style="간결하게")
make_expert_chain(domain="의료", style="쉽게 설명")
```

In [7]:
# ── 프롬프트 템플릿 팩토리 ────────────────────────────────────
def make_analyst_chain(domain: str, perspective: str, output_format: str):
    """
    재사용 가능한 분석 체인을 생성한다.

    Args:
        domain      : 전문 도메인 (예: '경영', '기술', '심리')
        perspective : 분석 관점 (예: '비용 중심', '사람 중심')
        output_format: 출력 형식 지시문
    """
    prompt = ChatPromptTemplate.from_messages([
        ("system", f"""당신은 {domain} 분야 전문 분석가입니다.
{perspective} 관점에서 분석하세요.

출력 형식:
{output_format}"""),
        ("human", "{question}")
    ])
    
    # 체인 생성
    chain = prompt | llm | StrOutputParser()
    return chain


# 경영 분석 체인
biz_chain = make_analyst_chain(
    domain="경영",
    perspective="ROI와 생산성 중심",
    output_format="[핵심 지표] → [예상 효과] → [실행 권고] (각 2~3줄)"
)

# 조직심리 분석 체인
psych_chain = make_analyst_chain(
    domain="조직심리",
    perspective="구성원 동기·몰입 중심",
    output_format="[심리적 영향] → [리스크] → [권고사항] (각 2~3줄)"
)

question = "재택근무 전면 도입, 찬성해야 할까요?"

print("=== 경영 분석 관점 ===")
print(biz_chain.invoke({"question": question}))

print("\n=== 조직심리 관점 ===")
print(psych_chain.invoke({"question": question}))

=== 경영 분석 관점 ===
[핵심 지표]  
사무실 임대·관리비 절감, 출퇴근 시간 감소에 따른 실근로시간 증가, 이직률·채용 경쟁력 변화가 핵심입니다.  
반면 협업 지연, 온보딩 효율 저하, 성과관리 난이도 상승으로 팀 생산성이 직무별로 크게 갈릴 수 있습니다.  

[예상 효과]  
개별 집중업무 비중이 높은 조직은 고정비 절감과 직원 만족도 상승으로 ROI가 개선될 가능성이 큽니다.  
하지만 창의 협업, 현장 대응, 신속한 의사결정이 중요한 조직은 커뮤니케이션 비용 증가로 생산성 저하가 발생할 수 있습니다.  

[실행 권고]  
전면 도입을 즉시 결정하기보다 직무별로 생산성 측정이 가능한 3~6개월 파일럿을 먼저 운영하는 것이 바람직합니다.  
회의시간, 프로젝트 리드타임, 이직률, 채용비용, 1인당 성과를 비교해 ROI가 확인되는 부문만 단계적으로 확대하는 방식을 권고합니다.

=== 조직심리 관점 ===
[심리적 영향]  
재택근무 전면 도입은 자율성과 시간 통제감을 높여 내재적 동기와 직무만족을 강화할 수 있습니다. 반면 소속감, 비공식 상호작용, 즉각적 피드백이 줄어들면 몰입과 팀 동일시가 약해질 수 있습니다.  

[리스크]  
구성원별로 자기조절 능력, 주거환경, 돌봄 부담 차이가 커서 동기 격차가 확대될 수 있습니다. 또한 성과평가 기준이 불명확하면 “보이지 않는 노동”에 대한 불안이 커지고, 장기적으로 고립감·번아웃이 누적될 가능성이 있습니다.  

[권고사항]  
전면 찬반보다 직무 특성과 구성원 준비도를 기준으로 단계적 도입을 권합니다. 목표·성과 기준을 명확히 하고, 정기적 1:1·팀 교류·오프라인 접점을 설계해 자율성과 연결감을 함께 관리하는 것이 바람직합니다.


---

## 7. 비교 실험: 같은 질문, 다른 기법

지금까지 배운 6가지 기법을 **같은 질문**에 적용해 결과를 비교합니다.  
답변의 구조, 깊이, 형식이 얼마나 다른지 확인하세요.

In [8]:
# ── 비교 실험: 기법별 답변 길이 & 구조 비교 ──────────────────
QUESTION = "재택근무 전면 도입, 찬성해야 할까요?"

# 각 기법의 체인 목록
techniques = [
    ("Zero-shot",         zero_shot_prompt),
    ("Few-shot",          few_shot_prompt),
    ("Chain-of-Thought",  cot_prompt),
]

print(f"질문: {QUESTION}")
print("=" * 60)

results = {}
for name, prompt in techniques:
    chain = prompt | llm | StrOutputParser()
    answer = chain.invoke({"question": QUESTION})
    results[name] = answer

# 결과 요약 출력
print(f"\n{'기법':<20} {'글자수':>8} {'줄 수':>6}")
print("-" * 38)
for name, answer in results.items():
    print(f"{name:<20} {len(answer):>8,} {len(answer.splitlines()):>6}")

질문: 재택근무 전면 도입, 찬성해야 할까요?

기법                        글자수    줄 수
--------------------------------------
Zero-shot                 932     37
Few-shot                  251     13
Chain-of-Thought          870     31


In [9]:
# ── 답변 전체 출력 ────────────────────────────────────────────
for name, answer in results.items():
    print(f"\n{'='*60}")
    print(f"[{name}]")
    print('='*60)
    # 처음 400자만 출력 (너무 길면 ...으로 줄임)
    preview = answer[:400]
    print(preview)
    if len(answer) > 400:
        print(f"\n... (총 {len(answer):,}자 중 400자 미리보기)")


[Zero-shot]
재택근무 전면 도입은 **무조건 찬성/반대**로 보기보다, **업무 성격과 조직 문화에 따라 신중히 판단해야 하는 사안**입니다. 다만 일반적으로는 **전면 도입보다는 유연한 혼합형(하이브리드) 방식이 더 현실적**이라고 볼 수 있습니다.

## 찬성하는 이유
1. **출퇴근 시간 절감**
   - 직원들의 피로가 줄고 삶의 질이 높아집니다.
   - 절약한 시간을 업무나 자기계발에 활용할 수 있습니다.

2. **업무 효율 향상 가능성**
   - 방해 요소가 적은 환경에서는 집중력이 높아질 수 있습니다.
   - 특히 문서 작성, 개발, 분석 같은 업무에 유리합니다.

3. **기업 비용 절감**
   - 사무실 임대료, 관리비, 운영비를 줄일 수 있습니다.

4. **인재 확보 범위 확대**
   - 

... (총 932자 중 400자 미리보기)

[Few-shot]
질문: 재택근무 전면 도입, 찬성해야 할까요?
답변:
✅ 찬성 근거
- 출퇴근 시간 절감으로 직원 만족도와 업무 집중도 향상
- 지역 제한 없는 인재 채용 가능
- 사무실 운영비 절감

❌ 반대 근거
- 대면 협업과 즉각적인 소통이 어려워질 수 있음
- 조직 문화 약화 및 소속감 저하 우려
- 직무에 따라 성과 관리와 보안 통제가 까다로움

💡 결론: 전면 도입보다는 직무별 적합성을 따져 하이브리드 근무부터 단계적으로 확대하는 것이 바람직합니다.

[Chain-of-Thought]
[1단계: 문제 파악]  
질문은 “재택근무를 전면 도입하는 것에 찬성할 것인가”에 대한 판단을 묻고 있습니다. 즉, 일부 허용이 아니라 **전면적·기본적 근무 방식으로 재택근무를 채택하는 것이 바람직한지**를 따져야 합니다.

[2단계: 이해관계자 분석]  
영향을 받는 주체는 다음과 같습니다.

- **직원**: 출퇴근 부담 감소, 일·생활 균형 향상 가능
- **기업 경영진**: 생산성, 비용 절감, 인력 관리 방식 변화
- **관리자**: 성과관리, 소통, 팀 운영 방식 재설계 필요

---

## 핵심 정리

```python
# 프롬프트 엔지니어링 선택 가이드

# 1. Zero-shot: 빠른 테스트, 일반 질문
chain = ChatPromptTemplate.from_messages([("human", "{q}")]) | llm | StrOutputParser()

# 2. Few-shot: 특정 형식 유도, 스타일 일관성
#    → 시스템 프롬프트에 예시 2~3개 포함

# 3. Chain-of-Thought: 복잡한 추론, 다단계 분석
#    → "단계별로 생각하세요" 지시문 추가

# 4. 역할 프롬프트: 전문 도메인, 특정 관점 필요 시
#    → 구체적일수록 좋음 ("전문가" < "10년 경력 변호사")

# 5. 구조화 출력: 파싱·연동이 필요한 경우
#    → JSON 형식 + "다른 텍스트 포함 금지" 명시

# 6. 팩토리 패턴: 동일 구조 반복 시
def make_chain(domain, style):
    prompt = ChatPromptTemplate.from_messages([...])
    return prompt | llm | StrOutputParser()
```

| 기법 | 최적 상황 | 주의사항 |
|---|---|---|
| Zero-shot | 범용 질문, 빠른 프로토타입 | 형식 보장 없음 |
| Few-shot | 특정 형식·스타일 필요 | 예시가 너무 많으면 토큰 낭비 |
| CoT | 복잡한 추론·판단 | 응답이 길어짐 |
| 역할 | 전문 도메인 | 구체적일수록 효과적 |
| 구조화 출력 | API 연동, 파싱 필요 | 100% 보장 안 됨 (검증 로직 필요) |
| 팩토리 | 반복 사용 패턴 | 과도한 추상화 주의 |
